# ApexTracking - Data Collection Notebook
Tento notebook je urceny pouze pro sber dat.
Obsahuje dva workflow: sber podle seznamu jmen a UID harvesting.

## 1) Importy a zavislosti
Nacteme knihovny a funkce z runtime vrstvy API.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import csv
import random
import time

from src.apex_api import (
    CollectorError,
    fetch_player_stats,
    fetch_player_stats_by_uid,
    is_row_usable,
    load_api_key,
    player_to_row,
)

## 2) Sber dat podle seznamu jmen
Tento blok se hodi, kdyz mas konkretni seznam hracu.
Kazdy profil se validuje a uklada jen pokud ma dostatecny signal dat.

In [ ]:
def collect_players_to_csv(
    players,
    out_csv='data/players.csv',
    platform='PC',
    sleep_seconds=0.25,
    min_level=25.0,
    min_kills=80.0,
    min_damage=20000.0,
    min_rank_score=1000.0,
    min_nonzero_metrics=3,
    allow_no_gameplay_signal=False,
):
    api_key = load_api_key()
    rows = []
    seen = set()

    for name in [str(p).strip() for p in players if str(p).strip()]:
        try:
            payload = fetch_player_stats(player=name, api_key=api_key, platform=platform)
            row = player_to_row(payload, requested_name=name)
            if not is_row_usable(
                row,
                min_level=min_level,
                min_kills=min_kills,
                min_damage=min_damage,
                min_rank_score=min_rank_score,
                min_nonzero_metrics=min_nonzero_metrics,
                require_gameplay_signal=not allow_no_gameplay_signal,
            ):
                continue

            key = str(row.get('uid', '')).strip() or str(row.get('player', '')).strip().lower()
            if key in seen:
                continue
            seen.add(key)
            rows.append(row)
            print(f'OK: {name}')
        except CollectorError as exc:
            print(f'SKIP {name}: {exc}')

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    if not rows:
        raise RuntimeError('No rows collected.')

    out = Path(out_csv)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {len(rows)} rows to {out}')
    return rows

## 3) UID harvesting workflow
Tento blok se hodi, kdyz chces rychle rozsirit dataset i bez dlouheho seznamu jmen.
Vytvari kandidaty UID kolem existujicich seed UID a paralelne je overuje pres API.

In [ ]:
def harvest_uids_to_csv(
    target=1500,
    max_attempts=250000,
    out_csv='data/players.csv',
    seed_csv='data/players.csv',
    platform='PC',
    workers=4,
    request_timeout=20.0,
    checkpoint_every=25,
    min_level=25.0,
    min_kills=80.0,
    min_damage=20000.0,
    min_rank_score=1000.0,
    min_nonzero_metrics=3,
    allow_no_gameplay_signal=False,
):
    api_key = load_api_key()
    out = Path(out_csv)

    seeds = []
    if Path(seed_csv).exists():
        with Path(seed_csv).open('r', newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                uid = str(r.get('uid', '')).strip()
                if uid.isdigit():
                    seeds.append(int(uid))

    if not seeds:
        seeds = [2796574388, 1008248071359, 1008995227775, 1003944652988]

    rows = []
    collected = set()
    seen_probe = set()
    attempts = 0

    def candidate_uid():
        pivot = random.choice(seeds)
        return str(max(1, pivot + random.randint(-25000, 25000)))

    def fetch_one(uid):
        try:
            payload = fetch_player_stats_by_uid(uid=uid, api_key=api_key, platform=platform, timeout=request_timeout)
            row = player_to_row(payload)
            if not is_row_usable(
                row,
                min_level=min_level,
                min_kills=min_kills,
                min_damage=min_damage,
                min_rank_score=min_rank_score,
                min_nonzero_metrics=min_nonzero_metrics,
                require_gameplay_signal=not allow_no_gameplay_signal,
            ):
                return None
            return row
        except Exception:
            return None

    with ThreadPoolExecutor(max_workers=max(1, int(workers))) as ex:
        while len(rows) < int(target) and attempts < int(max_attempts):
            batch = []
            while len(batch) < max(1, workers * 2) and attempts < int(max_attempts):
                uid = candidate_uid()
                attempts += 1
                if uid in seen_probe:
                    continue
                seen_probe.add(uid)
                batch.append(uid)

            futures = [ex.submit(fetch_one, uid) for uid in batch]
            for fu in as_completed(futures):
                row = fu.result()
                if row is None:
                    continue
                uid = str(row.get('uid', '')).strip()
                if not uid or uid in collected:
                    continue
                collected.add(uid)
                rows.append(row)
                seeds.append(int(uid))

                if len(rows) % checkpoint_every == 0:
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with out.open('w', newline='', encoding='utf-8') as f:
                        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
                        writer.writeheader()
                        writer.writerows(rows)
                    print(f'Checkpoint: {len(rows)} rows after {attempts} attempts')

            if len(rows) >= int(target):
                break

    if not rows:
        raise RuntimeError('No UID rows harvested.')

    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f'Harvest done: {len(rows)} rows after {attempts} attempts -> {out}')
    return rows

## 4) Priklady pouziti
Spust pouze jeden workflow podle toho, co zrovna potrebujes.

In [ ]:
# Priklad A: sber podle seznamu jmen
# players = ['ImperialHal', 'iiTzTimmy', 'Aceu']
# collect_players_to_csv(players, out_csv='data/players.csv')

# Priklad B: UID harvesting
# harvest_uids_to_csv(target=500, max_attempts=20000, workers=6, out_csv='data/players.csv')